In [1]:
"""
MULTI-MODEL EVALUATION SYSTEM
Supports:
1. Gas & Env model (environmental + gas sensors)
2. All model (environmental + gas + specter)
3. Specter model (spectral features only)

Each model predicts class 0–4.
Late fusion = simple average (no weights).
"""

import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

ModuleNotFoundError: No module named 'numpy'

In [ ]:
class SimpleModel:
    def __init__(self, name, features, model_path=None, scaler_path=None):
        self.name = name
        self.features = features
        self.model_path = model_path
        self.scaler_path = scaler_path

        # Load real model if paths are provided
        if model_path and scaler_path:
            try:
                print(f"🔹 Loading model for {name} from {model_path}")
                self.model = joblib.load(model_path)
                print(f"🔹 Loading scaler for {name} from {scaler_path}")
                self.scaler = joblib.load(scaler_path)
            except Exception as e:
                print(f"Failed to load {name} model/scaler ({e}). Using dummy model.")
                self._make_dummy_model()
        else:
            print(f"No paths provided for {name}. Using dummy model for testing.")
            self._make_dummy_model()

    def _make_dummy_model(self):
        """Creates a fake model for structural testing."""
        self.scaler = StandardScaler()
        X_dummy = np.random.rand(50, len(self.features))
        y_dummy = np.random.randint(0, 5, 50)
        X_dummy = self.scaler.fit_transform(X_dummy)
        self.model = RandomForestClassifier(random_state=42)
        self.model.fit(X_dummy, y_dummy)

    def predict(self, values):
        """Predict class (0–4) for a given input feature vector."""
        X = np.array(values).reshape(1, -1)
        X = self.scaler.transform(X)
        pred = int(self.model.predict(X)[0])
        return pred

In [ ]:
gas_env_features = ["Temp-int", "Press-int", "Humid-int", "Temp-ext", "Press-ext", "Humid-ext", "TGS20", "TGS02", "SGP"]
specter_features = ["SpA410","SpB435","SpC460","SpD485","SpE510","SpF535","SpG560","SpH585","SpR610","SpI645","SpS680","SpJ705","SpT730","SpU760","SpV810","SpW860","SpK900","SpL940"]
all_features = gas_env_features + specter_features

models = {
    "1": SimpleModel(
        "Gas & Env", gas_env_features,
        model_path="gas&env_models/random_forest_model_s.pkl",
        scaler_path="gas&env_models/random_forest_model_s.pkl"
    ),
    "2": SimpleModel(
        "All", all_features,
        model_path="all_models/random_forest_model_s.pkl",
        scaler_path="all_models/random_forest_model_s.pkl"
    ),
    "3": SimpleModel(
        "Specter", specter_features,
        model_path="specter_models/random_forest_model_s.pkl",
        scaler_path="specter_models/random_forest_model_s.pkl"
    )
}

In [ ]:

print("\nAvailable models:")
print("1 - Gas & Env")
print("2 - All")
print("3 - Specter")
chosen = input("\nEnter model numbers to evaluate (comma-separated, e.g., 1,2): ").split(",")

results = {}

for m in chosen:
    m = m.strip()
    if m not in models:
        print(f"Model {m} not found. Skipping.")
        continue
    
    model = models[m]
    print(f"\nEnter input for {model.name}:")
    user_input = []
    for feat in model.features:
        val = float(input(f"  {feat}: "))
        user_input.append(val)

    pred = model.predict(user_input)
    results[model.name] = pred

# -----------------------
# SHOW RESULTS
# -----------------------
if results:
    print("\nModel Predictions:")
    for name, pred in results.items():
        print(f"  {name}: Class {pred}")

    final = int(round(np.mean(list(results.values()))))
    print(f"\nFinal (Late Fusion) Result: Class {final}")
else:
    print("No valid models selected or predictions failed.")
